# A chatbot with two tools

**dabeer.ai · AI for Beginners** · AI agents, tools and MCP

You'll give a model two tools and watch the **tool-call loop** happen step by step:

1. You describe the tools: a name, a description and parameters.
2. The model decides whether it needs a tool, and **asks** for it.
3. **Your code** runs the tool and sends the result back.
4. The model writes the final answer, or asks for another tool.

The two tools are practice versions with made-up data:

- `get_weather`: the weather in an Indian city.
- `convert_currency`: converts between Indian rupees and a few other currencies, at fixed practice rates.

Run the cells from top to bottom with **Shift + Enter**.

### No paid API key? Use Gemini's free key

This notebook starts with `PROVIDER = "gemini"`, because Google AI Studio gives you a free Gemini API key, with no payment method needed.

1. Go to https://aistudio.google.com/apikey and sign in with your Google account.
2. Click **Create API key**, and copy the key.
3. In Colab, click the **key icon** in the left sidebar, add a secret named `GEMINI_API_KEY`, paste the key as the value, and switch on **Notebook access**.

Two things to know about the free key:

- It has rate limits. If you see a "429" or "resource exhausted" error, wait a minute and run the cell again.
- On the free tier, Google may use what you send, and the replies, to improve its products, and people may read them. So use only fictional or practice data, like the practice weather and currency data here. Never real customer, company or personal data.

Have a paid Anthropic or OpenAI key instead? Set `PROVIDER` to `"anthropic"` or `"openai"` in the Settings cell.

## Step 1: Install the libraries

In [ ]:
%pip install -q google-genai anthropic openai

## Step 2: Settings

- `PROVIDER`: `"gemini"` for Google's Gemini (free key), `"anthropic"` for Claude, or `"openai"` for ChatGPT models. If you have a second key, the lab asks you to try that provider too.
- `MAX_STEPS`: the step limit. An agent loop should never be allowed to run forever.

In [ ]:
PROVIDER = "gemini"                     # "gemini" (free key), "anthropic" or "openai"
GEMINI_MODEL = "gemini-3.1-flash-lite"       # model names change: pick a current one from the AI Studio list
ANTHROPIC_MODEL = "claude-sonnet-5-5"   # check the provider's model list for current names
OPENAI_MODEL = "gpt-5-mini"             # check the provider's model list for current names
MAX_STEPS = 5                           # most model calls allowed for one question

## Step 3: Your API key, from Colab Secrets

Click the **key icon** in Colab's left sidebar, add `GEMINI_API_KEY`, `ANTHROPIC_API_KEY` and/or `OPENAI_API_KEY`, and switch on **Notebook access**. Never type a key into a cell.

In [ ]:
from google.colab import userdata
from google import genai
from google.genai import types
import anthropic
from openai import OpenAI

def make_client(provider):
    if provider == "gemini":
        return genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
    if provider == "anthropic":
        return anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
    if provider == "openai":
        return OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
    raise ValueError("PROVIDER must be 'gemini', 'anthropic' or 'openai'")

client = make_client(PROVIDER)
print("Ready:", PROVIDER)

## Step 4: The two tools, as normal Python functions

This is the "kitchen". The model never runs these itself. Our code does.

Notice that each function **checks its inputs** before doing anything. The model chooses the inputs, so never trust them blindly.

In [ ]:
# Practice data only. Not real weather, not real exchange rates.
FAKE_WEATHER = {
    "delhi":     {"temp_c": 31, "condition": "humid, hazy"},
    "mumbai":    {"temp_c": 29, "condition": "light rain"},
    "bengaluru": {"temp_c": 24, "condition": "cloudy"},
    "chennai":   {"temp_c": 33, "condition": "sunny, humid"},
    "kolkata":   {"temp_c": 30, "condition": "thunderstorms likely"},
    "pune":      {"temp_c": 26, "condition": "partly cloudy"},
    "jaipur":    {"temp_c": 35, "condition": "hot and dry"},
}

# How many Indian rupees one unit of each currency is worth (practice rates).
INR_PER_UNIT = {"INR": 1.0, "USD": 85.0, "EUR": 92.0, "GBP": 108.0, "AED": 23.0, "SGD": 63.0, "JPY": 0.57}

def get_weather(city):
    key = str(city).strip().lower()
    if key not in FAKE_WEATHER:
        return {"error": f"No weather data for '{city}'. Known cities: {', '.join(c.title() for c in FAKE_WEATHER)}"}
    return {"city": key.title(), **FAKE_WEATHER[key], "note": "practice data"}

def convert_currency(amount, from_currency, to_currency):
    try:
        amount = float(amount)
    except (TypeError, ValueError):
        return {"error": "amount must be a number"}
    if amount < 0 or amount > 10_000_000:
        return {"error": "amount must be between 0 and 1 crore"}
    src, dst = str(from_currency).upper(), str(to_currency).upper()
    for code in (src, dst):
        if code not in INR_PER_UNIT:
            return {"error": f"Unknown currency '{code}'. Supported: {', '.join(INR_PER_UNIT)}"}
    result = amount * INR_PER_UNIT[src] / INR_PER_UNIT[dst]
    return {"amount": amount, "from": src, "to": dst, "result": round(result, 2), "note": "practice rate"}

# Try them directly, with no AI involved:
print(get_weather("Delhi"))
print(convert_currency(100, "USD", "INR"))
print(convert_currency(5000, "INR", "AED"))

## Step 5: Describe the tools (the "menu")

Each tool gets a **name**, a **description** in plain English, and **parameters** as a JSON schema. The model chooses tools by reading these, so write them for a new colleague.

The three providers want the same information in slightly different wrappers, so we write it once and convert it.

In [ ]:
TOOLS = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a city in India. Use this when the user asks about weather, temperature or rain.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "City name, for example Delhi or Pune"}},
            "required": ["city"],
        },
    },
    {
        "name": "convert_currency",
        "description": "Convert an amount of money between Indian rupees (INR) and another currency, such as USD, EUR, GBP, AED, SGD or JPY.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number", "description": "The amount to convert"},
                "from_currency": {"type": "string", "description": "Three-letter code to convert from, for example USD"},
                "to_currency": {"type": "string", "description": "Three-letter code to convert to, for example INR"},
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
]

# The same tools, in each provider's format
GEMINI_TOOLS = [types.Tool(function_declarations=TOOLS)]   # Gemini calls them "function declarations"
ANTHROPIC_TOOLS = [{"name": t["name"], "description": t["description"], "input_schema": t["parameters"]} for t in TOOLS]
OPENAI_TOOLS = [{"type": "function", "function": t} for t in TOOLS]

# Our code's lookup table: tool name -> Python function
TOOL_FUNCTIONS = {"get_weather": get_weather, "convert_currency": convert_currency}

def run_tool(name, args):
    if name not in TOOL_FUNCTIONS:
        return {"error": f"Unknown tool {name}"}
    try:
        return TOOL_FUNCTIONS[name](**args)
    except TypeError as e:          # wrong or missing arguments
        return {"error": str(e)}

## Step 6: The tool-call loop

This is the heart of an agent. Read the comments, then run the cell. Every step is printed so you can watch it work:

- `MODEL ASKS` is the model's structured request.
- `OUR CODE RETURNS` is what our function sent back.

In [ ]:
import json

SYSTEM = ("You are a helpful travel assistant for people in India. "
          "Use the tools when you need weather or currency numbers. Never guess them. "
          "Keep answers short.")

def ask_gemini(question, client, model, verbose=True):
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=question)])]
    config = types.GenerateContentConfig(system_instruction=SYSTEM, tools=GEMINI_TOOLS)
    for step in range(1, MAX_STEPS + 1):
        response = client.models.generate_content(model=model, contents=contents, config=config)
        if not response.function_calls:                 # no tool needed: this is the final answer
            return response.text
        contents.append(response.candidates[0].content)  # keep the model's request
        results = []
        for call in response.function_calls:
            args = dict(call.args or {})
            output = run_tool(call.name, args)
            if verbose:
                print(f"  step {step}  MODEL ASKS: {call.name}({args})")
                print(f"          OUR CODE RETURNS: {output}")
            results.append(types.Part.from_function_response(name=call.name, response=output))
        contents.append(types.Content(role="user", parts=results))    # send the results back
    return "(Stopped: reached the step limit)"

def ask_anthropic(question, client, model, verbose=True):
    messages = [{"role": "user", "content": question}]
    for step in range(1, MAX_STEPS + 1):
        response = client.messages.create(
            model=model, max_tokens=800, system=SYSTEM, tools=ANTHROPIC_TOOLS, messages=messages)
        if response.stop_reason != "tool_use":          # no tool needed: this is the final answer
            return "".join(b.text for b in response.content if b.type == "text")
        messages.append({"role": "assistant", "content": response.content})   # keep the model's request
        results = []
        for block in response.content:
            if block.type == "tool_use":
                output = run_tool(block.name, block.input)
                if verbose:
                    print(f"  step {step}  MODEL ASKS: {block.name}({block.input})")
                    print(f"          OUR CODE RETURNS: {output}")
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(output)})
        messages.append({"role": "user", "content": results})           # send the results back
    return "(Stopped: reached the step limit)"

def ask_openai(question, client, model, verbose=True):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for step in range(1, MAX_STEPS + 1):
        response = client.chat.completions.create(model=model, messages=messages, tools=OPENAI_TOOLS)
        message = response.choices[0].message
        if not message.tool_calls:                      # no tool needed: this is the final answer
            return message.content
        messages.append({"role": "assistant", "content": message.content,      # keep the model's request
                         "tool_calls": [{"id": c.id, "type": "function",
                                         "function": {"name": c.function.name, "arguments": c.function.arguments}}
                                        for c in message.tool_calls]})
        for call in message.tool_calls:
            args = json.loads(call.function.arguments)
            output = run_tool(call.function.name, args)
            if verbose:
                print(f"  step {step}  MODEL ASKS: {call.function.name}({args})")
                print(f"          OUR CODE RETURNS: {output}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(output)})   # send the result back
    return "(Stopped: reached the step limit)"

MODELS = {"gemini": GEMINI_MODEL, "anthropic": ANTHROPIC_MODEL, "openai": OPENAI_MODEL}
ASK_FUNCTIONS = {"gemini": ask_gemini, "anthropic": ask_anthropic, "openai": ask_openai}

def ask(question, provider=None):
    provider = provider or PROVIDER
    model = MODELS[provider]
    c = client if provider == PROVIDER else make_client(provider)
    print(f"YOU ({provider}): {question}")
    answer = ASK_FUNCTIONS[provider](question, c, model)
    print("ANSWER:", answer, "\n")
    return answer

ask("What's the weather in Delhi right now?")

## Step 7: Watch the model choose

Run these one at a time and look at the printed steps. Which tool did it pick? How many steps did it take?

In [ ]:
ask("How much is 250 US dollars in rupees?")

In [ ]:
ask("What is the capital of Karnataka?")      # no tool needed

In [ ]:
ask("I'm flying from Mumbai to Dubai with 40,000 rupees. How many dirhams is that, and should I pack an umbrella for Mumbai?")

In [ ]:
ask("What's the weather in Shimla?")          # not in our data: what does the model do with the error?

## Step 8: Same question, other provider

Add a second provider's key to Colab Secrets first, and set `OTHER` to that provider. Then run this. The tools and the loop are the same idea; only the wrapper differs.

In [ ]:
OTHER = "anthropic" if PROVIDER != "anthropic" else "gemini"   # any provider you have a second key for
ask("How much is 250 US dollars in rupees?", provider=OTHER)

## Step 9: A chat that remembers

So far each question started fresh. This version keeps the conversation, so you can ask follow-ups like "and in euros?". It uses the provider from Step 2. Type `quit` to stop.

In [ ]:
def chat():
    history = []
    while True:
        user = input("You: ").strip()
        if user.lower() in {"quit", "exit", ""}:
            break
        history.append(f"User: {user}")
        transcript = "\n".join(history[-10:])          # keep the last 10 turns
        answer = ASK_FUNCTIONS[PROVIDER]("Conversation so far:\n" + transcript + "\n\nReply to the last user message.", client, MODELS[PROVIDER])
        history.append(f"Assistant: {answer}")
        print("Bot:", answer, "\n")

chat()